In [1]:
import os
import json
import random 
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
from requests.exceptions import RequestException, Timeout, ConnectionError

# Initialize OpenAI client and load data
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Paths
base_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/pororo"
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

# Load dataset
with open(qa_json_path, "r") as f:
    qa_data = json.load(f)["PororoQA"]

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

# Multi agent

In [2]:
# Visual agent
def visual_agent(gif_paths, max_retries=3, retry_delay=2):
    """Process GIF and extract visual information"""
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())
            
    prompt = "Describe the visual content of these GIF frames in detail."

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=150,
                temperature=0.3,
            )
            return completion.choices[0].message.content.strip()

        except (RequestException, Timeout, ConnectionError) as e:
            wait_time = retry_delay * (2 ** attempt)
            print(f"Network error on attempt {attempt + 1}/{max_retries}: {e}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            
    return None

# Language agent 
def language_agent(question, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    """Generate answer based on question and all available information"""
    prompt = f"""
    Question: {question}
    Scene Description: {description}
    Visual Description: {visual_desc}
    Subtitles: {subtitles}
    
    Generate a detailed answer based on all the available information.
    """
    
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=150,
                temperature=0.3,
            )
            return completion.choices[0].message.content.strip()

        except (RequestException, Timeout, ConnectionError) as e:
            wait_time = retry_delay * (2 ** attempt)
            print(f"Network error on attempt {attempt + 1}/{max_retries}: {e}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            
    return None

# Hallucination detection agent
def hallucination_agent(question, answer, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    """Detect and correct potential hallucinations in the answer"""
    prompt = f"""
    Question: {question}
    Scene Description: {description}
    Visual Description: {visual_desc}
    Subtitles: {subtitles}
    Generated Answer: {answer}
    
    Verify if the generated answer is accurate and supported by the available information.
    If hallucinations are detected, provide a corrected answer.
    """

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini", 
                messages=[{"role": "user", "content": prompt}],
                max_tokens=150,
                temperature=0.3,
            )
            return completion.choices[0].message.content.strip()

        except (RequestException, Timeout, ConnectionError) as e:
            wait_time = retry_delay * (2 ** attempt)
            print(f"Network error on attempt {attempt + 1}/{max_retries}: {e}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            
    return None

# Check predicted answer 

In [3]:
def check_answer(correct_answer, predicted_answer):
    """Compare predicted answer with ground truth"""
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.3
    )
    
    try:
        score = float(completion.choices[0].message.content.strip())
        valid_scores = [0.0, 0.25, 0.5, 0.75, 1.0]
        return min(valid_scores, key=lambda x: abs(x - score))
    except ValueError:
        return 0.0

# process data

In [4]:
def get_seeded_question(questions, gif_num, base_seed=42):
    """Get deterministic random question for a GIF"""
    if not questions:
        return None
    local_random = random.Random(base_seed + gif_num)
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

# Process data and calculate accuracy
random.seed(42)
evaluation_results = []

# TODO 修改，增加别的video
# Filter questions for episode 1
ep1_questions = [q for q in qa_data if "Pororo_ENGLISH1_1_ep1" in q["video_name"]]

# Group questions by supporting_num
grouped_questions = {}
for entry in ep1_questions:
    supporting_num = entry["supporting_num"]
    if supporting_num not in grouped_questions:
        grouped_questions[supporting_num] = []
    grouped_questions[supporting_num].append(entry)

# Process GIFs 1-36
gif_numbers = list(range(1, 37))
correct_count = 0
total_count = len(gif_numbers)
gif_accuracy = {}

# Process each GIF
for gif_num in tqdm(gif_numbers, total=total_count):
    current_gif_questions = grouped_questions.get(str(gif_num), [])
    
    if not current_gif_questions:
        print(f"No questions found for GIF {gif_num}")
        continue
        
    entry = get_seeded_question(current_gif_questions, gif_num)
    
    video_name = entry["video_name"]
    question = entry["question"] 
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Get paths and data
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", "Pororo_ENGLISH1_1", "Pororo_ENGLISH1_1_ep1")
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    gif_files = [f"{gif_num}.gif"]
    gif_paths = [os.path.join(episode_folder, gif_file) for gif_file in gif_files]

    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    # Multi-agent process
    visual_desc = visual_agent(gif_paths)
    initial_answer = language_agent(question, visual_desc, description, subtitles)
    final_answer = hallucination_agent(question, initial_answer, visual_desc, description, subtitles)
    
    # Use final answer or fall back to initial if hallucination check fails
    predicted_answer = final_answer if final_answer else initial_answer
    
    # Calculate accuracy
    is_correct = predicted_answer is not None and check_answer(correct_answer, predicted_answer)
    correct_count += is_correct

    # Store result
    result = {
        'gif_num': gif_num,
        'video_name': video_name,
        'qid': qid,
        'question': question,
        'correct_answer': correct_answer,
        'predicted_answer': predicted_answer,
        'accuracy': (is_correct * 100)
    }
    evaluation_results.append(result)

    # Print results
    print(f"Video name: {video_name}")
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"GIF number: {gif_num}, Accuracy: {(is_correct * 100):.0f}%")

# Calculate average accuracy
average_accuracy = correct_count / total_count
print(f"Average Accuracy: {average_accuracy * 100:.2f}%")



  3%|▎         | 1/36 [00:07<04:33,  7.81s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 1
QID: 4
Question: what is the nature of place where pororo leaves?
Correct Answer: it is the cold country , small village
Predicted Answer: The generated answer accurately reflects the information provided in the scene description and subtitles. It captures the essence of Pororo's environment, noting the cold climate, the small village in the forest, and the snow-covered mountains. The description of Pororo as a curious little penguin who encounters a baby dinosaur is also consistent with the narrative.

There are no hallucinations detected in the generated answer, as it aligns well with the details given in the scene description and subtitles. The answer effectively conveys the whimsical and adventurous nature of Pororo's world, highlighting both the serene setting and the playful storyline involving his new friend, the baby dinosaur. 

Overall, the generated answer is accurate and well-supported by the available information.
GIF number: 

  6%|▌         | 2/36 [00:16<04:50,  8.54s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 2
QID: 83
Question: what is the name of the little penguin
Correct Answer: the name of the little penguin is pororo
Predicted Answer: The generated answer is accurate and well-supported by the information provided in the scene description and subtitles. It correctly identifies the little penguin as Pororo and describes his environment as a cold village in a snowy forest. The answer also accurately recounts the storyline, where Pororo discovers an egg that hatches into a baby dinosaur named Crong, initially causing Pororo to think of it as a scary monster. 

Furthermore, the answer captures the essence of their developing friendship, highlighting their playful interactions and the counting exercise they engage in together. There are no hallucinations detected in the generated answer, and it effectively summarizes the key elements of the scene.
GIF number: 2, Accuracy: 100%


  8%|▊         | 3/36 [00:21<03:44,  6.79s/it]

...

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 34
QID: 655
Question: what phrase do all of the animals say to the green dinosaur crung?
Correct Answer: they all say, "nice to meet you".
Predicted Answer: The generated answer is mostly accurate and aligns with the information provided in the subtitles. The phrase that the animals say to the green dinosaur Crong is indeed "crong," which is a playful sound associated with Crong throughout the interaction. The answer captures the friendly and light-hearted nature of the scene, where Pororo, the penguin, meets Crong, the baby dinosaur.

However, the generated answer could be more concise and focused on the specific question about the phrase. Here’s a corrected and streamlined version:

In the scene described, the phrase that all of the animals say to the green dinosaur named Crong is "crong." This playful sound is repeated throughout the interaction, highlighting the friendly relationship that develops between Pororo, the curious little
GIF 

 97%|█████████▋| 35/36 [04:03<00:07,  7.23s/it]

Video name: Pororo_ENGLISH1_1_ep10
GIF number: 35
QID: 1119
Question: was pororo enjoying the new ride?
Correct Answer: yes, pororo was enjoying and said " yahoo"
Predicted Answer: The generated answer is mostly accurate in summarizing the scene and the interactions between Pororo and Crong. However, it does not address the specific question about whether Pororo was enjoying a new ride, as there is no mention of a ride in the provided scene description or subtitles. 

Instead, the focus is on Pororo's initial fear of the baby dinosaur and their subsequent friendship, which leads to playful interactions. While Pororo does seem to enjoy his time with Crong, there is no indication of a ride being part of the scene.

To correct the answer: 

Pororo is not depicted as enjoying a new ride in the provided scene. The focus is on his initial surprise and fear when the baby dinosaur hatches from the egg, followed
GIF number: 35, Accuracy: 25%


100%|██████████| 36/36 [04:10<00:00,  6.94s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 36
QID: 672
Question: who did pororo become friends with today?
Correct Answer: he became friends with baby dinosaur, crong
Predicted Answer: The generated answer is accurate and well-supported by the information provided in the scene description and subtitles. It correctly identifies that Pororo, the little penguin, becomes friends with a baby dinosaur named Crong. The answer captures the initial tension when Pororo thinks Crong is a scary monster after hatching from the egg, and it highlights their playful interaction that leads to the formation of their friendship.

There are no hallucinations detected in the generated answer, and it effectively summarizes the key points of the scene, including the setting and the characters' interactions. Therefore, the answer can be considered correct as it aligns with the details provided.
GIF number: 36, Accuracy: 100%
Average Accuracy: 60.42%


# Save results

In [13]:
# Remove any existing Average rows
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average Accuracy']

# Add average accuracy row using the average_accuracy from main processing
average_result = {
    'gif_num': 'Average Accuracy',  
    'video_name': '',
    'qid': '',
    'question': '',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy * 100  
}
evaluation_results.append(average_result)

# Save to CSV 
output_path = os.path.join("/Users/wt/PythonProjects/MultimodalComicAgent/results", 
                          'pororo_evaluation_results_multi_agent.csv')

# Save as DataFrame and force overwrite
results_df = pd.DataFrame(evaluation_results)
results_df.to_csv(output_path, index=False, mode='w')
print(f"Results saved to: {output_path}")
print(f"Total questions: {len(evaluation_results)-1}")  
print(f"Average accuracy: {average_accuracy*100:.2f}%")

Results saved to: /Users/wt/PythonProjects/MultimodalComicAgent/results/pororo_evaluation_results_multi_agent.csv
Total questions: 36
Average accuracy: 60.42%
